In [1]:
# ==========================================
# 0. UNIVERSAL DEPENDENCY INSTALLATION
# ==========================================
import subprocess
import sys

def install_dependencies():
    packages = [
        "torch", "transformers", "accelerate", "sentencepiece", "protobuf",
        "scikit-learn", "scipy", "pandas", "numpy"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_dependencies()

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
import gc

print("Loading Level 1 Core Political data...")
df = pd.read_csv('/kaggle/input/notebooks/shivanguniyal/level-1-theme-breakdown/FINAL_Level1_CorePolitical_2024.csv')
text_col = 'clean_text' if 'clean_text' in df.columns else 'article_text'

# Assign a unique Article ID for traceability
df['article_id'] = df.index

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("zero-shot-classification", 
                model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", 
                device=device, batch_size=32)

candidate_labels = ["critical", "neutral", "supportive"]
hypothesis_template = "The overall evaluative orientation conveyed by this news article is {}."
label_to_idx = {label: idx for idx, label in enumerate(candidate_labels)}

# ==========================================
# STAGE 1: TRUNCATED NLI (PRIMARY CLASSIFIER)
# ==========================================
print("\n--- STAGE 1: Fast Truncated Inference ---")
texts_trunc = [str(t)[:1500] for t in df[text_col].tolist()]

# Initialize NumPy array for fast probability storage
trunc_probs = np.zeros((len(df), 3))

super_chunk_size = 5000
for i in range(0, len(texts_trunc), super_chunk_size):
    batch = texts_trunc[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    
    for j, r in enumerate(res):
        idx = i + j
        for label, score in zip(r['labels'], r['scores']):
            trunc_probs[idx, label_to_idx[label]] = score
            
    print(f"  Stage 1 Progress: {min(i+super_chunk_size, len(texts_trunc)):,}/{len(texts_trunc):,}")

# Vectorized assignment to DataFrame (Instantaneous)
df['trunc_critical'] = trunc_probs[:, label_to_idx['critical']]
df['trunc_neutral'] = trunc_probs[:, label_to_idx['neutral']]
df['trunc_supportive'] = trunc_probs[:, label_to_idx['supportive']]
df['trunc_label'] = [candidate_labels[idx] for idx in np.argmax(trunc_probs, axis=1)]
df['trunc_confidence'] = np.max(trunc_probs, axis=1)

# ==========================================
# STAGE 2: EMPIRICAL ESCALATION RULE & METADATA
# ==========================================
print("\n--- STAGE 2: Identifying Uncertain Articles ---")
CONFIDENCE_THRESHOLD = 0.65 

escalation_mask = df['trunc_confidence'] < CONFIDENCE_THRESHOLD

# Add explicit boolean flag for pipeline diagnostics
df['escalated'] = escalation_mask 

num_to_rerun = escalation_mask.sum()
print(f"✓ {num_to_rerun:,} articles ({num_to_rerun/len(df)*100:.1f}%) flagged for Hierarchical Escalation.")

# ==========================================
# STAGE 3: HIERARCHICAL CLASSIFICATION (ADJUDICATION)
# ==========================================
def get_hierarchical_chunks(text, chunk_size=1000):
    text = str(text)
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    chunks, current_chunk = [], ""
    for sent in sentences:
        if len(current_chunk) + len(sent) > chunk_size:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sent
        else:
            current_chunk += " " + sent if current_chunk else sent
    if current_chunk: chunks.append(current_chunk)
    return chunks if chunks else [text]

# Initialize Hierarchical Columns with NaN
df['hier_critical'] = np.nan
df['hier_neutral'] = np.nan
df['hier_supportive'] = np.nan
df['hier_label'] = np.nan

if num_to_rerun > 0:
    print(f"\n--- STAGE 3: Hierarchical Adjudication on {num_to_rerun:,} articles ---")
    df_ambig = df[escalation_mask].copy()
    df_ambig['chunks'] = df_ambig[text_col].apply(get_hierarchical_chunks)
    
    all_chunks = []
    chunk_mapping = [] 
    for i, (idx, row) in enumerate(df_ambig.iterrows()):
        for chunk in row['chunks']:
            all_chunks.append(chunk)
            chunk_mapping.append(idx) 

    print(f"  Processing {len(all_chunks):,} total chunks...")
    
    # Accumulators for Mean Pooling
    hier_sums = {idx: np.zeros(3) for idx in df_ambig.index}
    hier_counts = {idx: 0 for idx in df_ambig.index}

    for i in range(0, len(all_chunks), super_chunk_size):
        batch = all_chunks[i:i+super_chunk_size]
        res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True, batch_size=16) 
        
        for j, r in enumerate(res):
            orig_idx = chunk_mapping[i + j]
            for label, score in zip(r['labels'], r['scores']):
                hier_sums[orig_idx][label_to_idx[label]] += score
            hier_counts[orig_idx] += 1
            
        print(f"  Stage 3 Progress: {min(i+super_chunk_size, len(all_chunks)):,}/{len(all_chunks):,}")

    # Vectorized write-back to main DataFrame
    hier_probs = np.full((len(df), 3), np.nan)
    hier_labels_arr = np.full(len(df), None, dtype=object)
    
    for orig_idx in df_ambig.index:
        avg_probs = hier_sums[orig_idx] / hier_counts[orig_idx]
        hier_probs[orig_idx] = avg_probs
        hier_labels_arr[orig_idx] = candidate_labels[np.argmax(avg_probs)]

    df['hier_critical'] = hier_probs[:, label_to_idx['critical']]
    df['hier_neutral'] = hier_probs[:, label_to_idx['neutral']]
    df['hier_supportive'] = hier_probs[:, label_to_idx['supportive']]
    df['hier_label'] = hier_labels_arr

# ==========================================
# STAGE 4: CONSOLIDATION & EXPORT
# ==========================================
print("\n--- STAGE 4: Consolidating Final Labels ---")

# Final Label: Hierarchical if escalated, otherwise Truncated
df['final_label'] = df['trunc_label']
df.loc[escalation_mask, 'final_label'] = df.loc[escalation_mask, 'hier_label']

# Track the method used and calculate disagreement flag
df['inference_method'] = 'Truncated (Primary)'
df.loc[escalation_mask, 'inference_method'] = 'Hierarchical (Adjudicated)'
df['disagreement'] = df['escalated'] & (df['trunc_label'] != df['hier_label'])

# ==========================================
# GENERATE STRATEGIC SPOT-CHECK
# ==========================================
print("\n--- Generating Blinded Spot Check ---")
# Over-sample the adjudicated/disagreement articles
spot_check_hier = df[df['escalated']].sample(min(50, num_to_rerun), random_state=42) if num_to_rerun > 0 else pd.DataFrame()
spot_check_trunc = df[~df['escalated']].groupby('final_label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 17), random_state=42)
)
spot_check = pd.concat([spot_check_hier, spot_check_trunc]).sample(frac=1, random_state=42)

cols_to_keep_spot = ['article_id', 'date', 'newspaper', 'period', 'Topic', text_col]
spot_check_blinded = spot_check[[c for c in cols_to_keep_spot if c in spot_check.columns]].copy()
spot_check_blinded['Human_Label'] = ""
spot_check_blinded['Coder_Notes'] = ""
spot_check_blinded.to_csv('/kaggle/working/English_Spot_Check_Blinded_2024.csv', index=False)

# ==========================================
# SAVE MASTER DATASET
# ==========================================
# Drop heavy text/chunk columns, keep ALL metadata and probabilities
cols_to_drop = [text_col, 'chunks'] if 'chunks' in df.columns else [text_col]
df_master = df.drop(columns=cols_to_drop)

# Reorder columns to match the Reviewer's requested metadata table
priority_cols = [
    'article_id', 'date', 'newspaper', 'period', 'Topic', 'Macro_Bucket',
    'trunc_label', 'trunc_critical', 'trunc_neutral', 'trunc_supportive', 'trunc_confidence',
    'escalated', 'hier_label', 'hier_critical', 'hier_neutral', 'hier_supportive',
    'final_label', 'inference_method', 'disagreement'
]
# Add any remaining columns that weren't in the priority list
remaining_cols = [c for c in df_master.columns if c not in priority_cols]
df_master = df_master[priority_cols + remaining_cols]

df_master.to_csv('/kaggle/working/English_Master_Processed_Level1_2024.csv', index=False)

# ==========================================
# PRINT FINAL SUMMARY
# ==========================================
print("\n" + "="*60)
print("📊 CASCADE INFERENCE SUMMARY")
print("="*60)
print(f"Total Articles: {len(df):,}")
print(f"Resolved via Truncated (High Confidence): {(~escalation_mask).sum():,}")
print(f"Escalated to Hierarchical (Uncertain): {num_to_rerun:,}")

if num_to_rerun > 0:
    disagreements = df['disagreement'].sum()
    print(f"Disagreements (Truncated vs Hierarchical): {disagreements:,} ({disagreements/num_to_rerun*100:.1f}% of escalated)")
print("="*60)

del pipe
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Pipeline complete! Ready for the Morning-After Diagnostics.")

Loading Level 1 Core Political data...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


--- STAGE 1: Fast Truncated Inference ---
  Stage 1 Progress: 5,000/248,365
  Stage 1 Progress: 10,000/248,365
  Stage 1 Progress: 15,000/248,365
  Stage 1 Progress: 20,000/248,365
  Stage 1 Progress: 25,000/248,365
  Stage 1 Progress: 30,000/248,365
  Stage 1 Progress: 35,000/248,365
  Stage 1 Progress: 40,000/248,365
  Stage 1 Progress: 45,000/248,365


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Stage 1 Progress: 50,000/248,365
  Stage 1 Progress: 55,000/248,365
  Stage 1 Progress: 60,000/248,365
  Stage 1 Progress: 65,000/248,365
  Stage 1 Progress: 70,000/248,365
  Stage 1 Progress: 75,000/248,365
  Stage 1 Progress: 80,000/248,365
  Stage 1 Progress: 85,000/248,365
  Stage 1 Progress: 90,000/248,365
  Stage 1 Progress: 95,000/248,365
  Stage 1 Progress: 100,000/248,365
  Stage 1 Progress: 105,000/248,365
  Stage 1 Progress: 110,000/248,365
  Stage 1 Progress: 115,000/248,365
  Stage 1 Progress: 120,000/248,365
  Stage 1 Progress: 125,000/248,365
  Stage 1 Progress: 130,000/248,365
  Stage 1 Progress: 135,000/248,365
  Stage 1 Progress: 140,000/248,365
  Stage 1 Progress: 145,000/248,365
  Stage 1 Progress: 150,000/248,365
  Stage 1 Progress: 155,000/248,365
  Stage 1 Progress: 160,000/248,365
  Stage 1 Progress: 165,000/248,365
  Stage 1 Progress: 170,000/248,365
  Stage 1 Progress: 175,000/248,365
  Stage 1 Progress: 180,000/248,365
  Stage 1 Progress: 185,000/248,365
  

/tmp/ipykernel_23/3418569453.py:171: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spot_check_trunc = df[~df['escalated']].groupby('final_label', group_keys=False).apply(



📊 CASCADE INFERENCE SUMMARY
Total Articles: 248,365
Resolved via Truncated (High Confidence): 99,887
Escalated to Hierarchical (Uncertain): 148,478
Disagreements (Truncated vs Hierarchical): 48,470 (32.6% of escalated)

✓ Pipeline complete! Ready for the Morning-After Diagnostics.
